<a href="https://colab.research.google.com/github/samar8x/CVIP_Project/blob/main/Notebooks/10_Modified_Model_Evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# 10 — Original vs Diagnosis-Driven Modified SegFormer-B2

## Goal
Evaluate the **original fixed SegFormer-B2** and the **modified SegFormer-B2 from Notebook 09** on the **untouched official ACDC validation split**.

This notebook answers:

> After diagnosing which classes changed most, which differences were reliable, and which classes had enough support, did the diagnosis-driven model modification actually improve those weaknesses?





## GPU setup



In [ ]:

import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA GPU is required. In Colab choose "
        "Runtime → Change runtime type → Hardware accelerator → GPU."
    )

DEVICE = torch.device("cuda")
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print("CUDA version:", torch.version.cuda)

torch.backends.cudnn.benchmark = True


CUDA available: True
GPU: Tesla T4
CUDA version: 13.0


In [ ]:

# Install/import dependencies
!pip -q install -U transformers accelerate safetensors

import os
import gc
import json
import math
import random
import shutil
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F

from transformers import AutoImageProcessor, SegformerForSemanticSegmentation

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 100)

print("PyTorch:", torch.__version__)


PyTorch: 2.14.1+cu130


In [ ]:

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:

# Project paths and constants
PROJECT_ROOT = '/content/drive/MyDrive/CV_Project/'
DATASETS_DIR = '/content/drive/MyDrive/CV_Project/datasets'
RESULTS_DIR = '/content/drive/MyDrive/CV_Project/results'
MODELS_DIR = '/content/drive/MyDrive/CV_Project/models'
MOD_RESULTS_DIR = '/content/drive/MyDrive/CV_Project/results/model_modification'
EVAL_RESULTS_DIR =  '/content/drive/MyDrive/CV_Project/results/model_modification/validation_evaluation'

PER_IMAGE_CACHE_DIR = '/content/drive/MyDrive/CV_Project/results/model_modification/validation_evaluation/per_image_confusions_v1'

MODEL_DIR =  '/content/drive/MyDrive/CV_Project/models/vulnerability_aware_segformer_b2'
STATE_PATH = '/content/drive/MyDrive/CV_Project/models/vulnerability_aware_segformer_b2/model_state_dict.pt'
MOD_CONFIG_PATH = '/content/drive/MyDrive/CV_Project/models/vulnerability_aware_segformer_b2/modification_config.json'
WEIGHTS_CSV = '/content/drive/MyDrive/CV_Project/models/vulnerability_aware_segformer_b2/vulnerability_aware_loss_weights.csv'

ACDC_RGB_ZIP = '/content/drive/MyDrive/CV_Project/datasets/acdc/rgb_anon.zip'
ACDC_GT_ZIP = '/content/drive/MyDrive/CV_Project/datasets/archives/gt_trainval.zip'

BASE_CHECKPOINT = 'nvidia/segformer-b2-finetuned-cityscapes-1024-1024'

CONDITIONS = ['fog', 'rain', 'snow', 'night']
NUM_CLASSES = 19
IGNORE_INDEX = 255

CLASS_NAMES = [
    'road', 'sidewalk', 'building', 'wall', 'fence', 'pole',
    'traffic light', 'traffic sign', 'vegetation', 'terrain', 'sky',
    'person', 'rider', 'car', 'truck', 'bus', 'train', 'motorcycle', 'bicycle'
]

STATIC_THIN = ['pole', 'traffic light', 'traffic sign']
DYNAMIC_SMALL = ['person', 'rider', 'motorcycle', 'bicycle']
LARGE_STRUCTURAL = ['road', 'building', 'vegetation', 'terrain', 'sky']

GROUPS = {
    'Static Thin': STATIC_THIN,
    'Dynamic Small': DYNAMIC_SMALL,
    'Large/Structural': LARGE_STRUCTURAL,
}

CLASS_TO_ID = {name: i for i, name in enumerate(CLASS_NAMES)}



print("Project root:", PROJECT_ROOT)
print("Modified checkpoint:", STATE_PATH)
print("Evaluation output:", EVAL_RESULTS_DIR)


Project root: /content/drive/MyDrive/CV_Project/
Modified checkpoint: /content/drive/MyDrive/CV_Project/models/vulnerability_aware_segformer_b2/model_state_dict.pt
Evaluation output: /content/drive/MyDrive/CV_Project/results/model_modification/validation_evaluation



## 1. Verify Notebook 09 output




In [ ]:

with open(MOD_CONFIG_PATH, 'r') as f:
    mod_config = json.load(f)

display(pd.json_normalize(mod_config, sep='.'))

assert mod_config['trainable_parts']['decode_head'] is True
assert mod_config['trainable_parts']['last_encoder_stage'] is True
assert mod_config['training']['official_acdc_validation_used'] is False

print("✓ Corrected Notebook 09 configuration verified.")
print("Best internal ACDC-train holdout mIoU:",
      mod_config['training']['best_internal_holdout_miou'])


In [ ]:

# Load the diagnosis-driven weights used in Notebook 09.
# These are NOT used to score validation; they only identify which condition/classes
# the modified training objective emphasized.
weight_df = pd.read_csv(WEIGHTS_CSV)

required_cols = {'condition', 'class_id', 'class_name', 'loss_weight'}
assert required_cols.issubset(weight_df.columns)

targeted_df = (
    weight_df
    .sort_values(['condition', 'loss_weight'], ascending=[True, False])
    .groupby('condition', as_index=False)
    .head(6)
    [['condition', 'class_name', 'matched_change',
      'positive_resample_fraction', 'pairs_with_class_both', 'loss_weight']]
)

print("Top diagnosis-driven training targets by condition:")
display(targeted_df)



## 2. Copy ACDC ZIP files to local Colab storage



In [ ]:

LOCAL_RGB_ZIP = Path('/content/acdc_rgb_anon_eval.zip')
LOCAL_GT_ZIP = Path('/content/acdc_gt_trainval_eval.zip')

def copy_if_needed(src, dst):
    if not dst.exists() or dst.stat().st_size != src.stat().st_size:
        print(f"Copying {src.name} to local storage...")
        shutil.copy2(src, dst)
    else:
        print(f"Local copy already exists: {dst}")
    print(dst, f"{dst.stat().st_size / 1024**3:.2f} GB")

copy_if_needed(ACDC_RGB_ZIP, LOCAL_RGB_ZIP)
copy_if_needed(ACDC_GT_ZIP, LOCAL_GT_ZIP)



## 3. Build the untouched ACDC validation manifest




In [ ]:

def scene_key_from_rgb(name):
    return Path(name).name.replace('_rgb_anon.png', '')

def scene_key_from_gt(name, suffix):
    return Path(name).name.replace(suffix, '')

def contains_exact_part(name, part):
    return part in Path(name).parts

with zipfile.ZipFile(LOCAL_RGB_ZIP, 'r') as z:
    rgb_names = [
        n for n in z.namelist()
        if n.endswith('_rgb_anon.png') and '__MACOSX' not in n
    ]

with zipfile.ZipFile(LOCAL_GT_ZIP, 'r') as z:
    gt_names = [
        n for n in z.namelist()
        if n.endswith('.png') and '__MACOSX' not in n
    ]

rgb_lookup, gt_lookup, inv_lookup = {}, {}, {}

for name in rgb_names:
    parts = Path(name).parts
    condition = next((c for c in CONDITIONS if c in parts), None)
    if condition is None:
        continue
    # Official adverse validation only; exclude val_ref.
    if 'val' in parts and 'val_ref' not in parts:
        key = scene_key_from_rgb(name)
        rgb_lookup[(condition, key)] = name

for name in gt_names:
    parts = Path(name).parts
    condition = next((c for c in CONDITIONS if c in parts), None)
    if condition is None:
        continue
    if 'val' not in parts or 'val_ref' in parts:
        continue

    if name.endswith('_gt_labelTrainIds.png'):
        key = scene_key_from_gt(name, '_gt_labelTrainIds.png')
        gt_lookup[(condition, key)] = name
    elif name.endswith('_gt_invIds.png'):
        key = scene_key_from_gt(name, '_gt_invIds.png')
        inv_lookup[(condition, key)] = name

rows = []
for (condition, key), rgb_member in sorted(rgb_lookup.items()):
    if (condition, key) in gt_lookup and (condition, key) in inv_lookup:
        rows.append({
            'condition': condition,
            'scene_key': key,
            'rgb_member': rgb_member,
            'gt_member': gt_lookup[(condition, key)],
            'inv_member': inv_lookup[(condition, key)],
        })

val_manifest = pd.DataFrame(rows).sort_values(['condition', 'scene_key']).reset_index(drop=True)

counts = val_manifest.groupby('condition').size().reindex(CONDITIONS)
display(counts.rename('validation_images').to_frame())

assert len(val_manifest) == 406, f"Expected 406 validation images, found {len(val_manifest)}"
assert counts.to_dict() == {'fog': 100, 'rain': 100, 'snow': 100, 'night': 106}

val_manifest.to_csv(EVAL_RESULTS_DIR / 'acdc_validation_manifest.csv', index=False)
print("✓ Untouched ACDC validation manifest verified.")


In [ ]:

# Extract only the 406 validation RGB/GT/invIds files to local storage
LOCAL_VAL_ROOT = Path('/content/acdc_validation_model_comparison')
RGB_ROOT = LOCAL_VAL_ROOT / 'rgb'
GT_ROOT = LOCAL_VAL_ROOT / 'gt'
INV_ROOT = LOCAL_VAL_ROOT / 'inv'

for d in [RGB_ROOT, GT_ROOT, INV_ROOT]:
    d.mkdir(parents=True, exist_ok=True)

def safe_local_name(condition, key):
    return f'{condition}__{key}.png'

with zipfile.ZipFile(LOCAL_RGB_ZIP, 'r') as z_rgb, zipfile.ZipFile(LOCAL_GT_ZIP, 'r') as z_gt:
    for i, row in tqdm(val_manifest.iterrows(), total=len(val_manifest), desc='Extracting validation'):
        out_rgb = RGB_ROOT / safe_local_name(row.condition, row.scene_key)
        out_gt = GT_ROOT / safe_local_name(row.condition, row.scene_key)
        out_inv = INV_ROOT / safe_local_name(row.condition, row.scene_key)

        if not out_rgb.exists():
            with z_rgb.open(row.rgb_member) as src, open(out_rgb, 'wb') as dst:
                shutil.copyfileobj(src, dst)
        if not out_gt.exists():
            with z_gt.open(row.gt_member) as src, open(out_gt, 'wb') as dst:
                shutil.copyfileobj(src, dst)
        if not out_inv.exists():
            with z_gt.open(row.inv_member) as src, open(out_inv, 'wb') as dst:
                shutil.copyfileobj(src, dst)

val_manifest['rgb_path'] = [
    str(RGB_ROOT / safe_local_name(r.condition, r.scene_key))
    for r in val_manifest.itertuples()
]
val_manifest['gt_path'] = [
    str(GT_ROOT / safe_local_name(r.condition, r.scene_key))
    for r in val_manifest.itertuples()
]
val_manifest['inv_path'] = [
    str(INV_ROOT / safe_local_name(r.condition, r.scene_key))
    for r in val_manifest.itertuples()
]

print("✓ Local validation files ready.")



## 4. Recreate the exact modified classifier from Notebook 09


In [ ]:

class ResidualChannelAttention(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        hidden = max(channels // reduction, 8)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.gate = nn.Sequential(
            nn.Conv2d(channels, hidden, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(hidden, channels, kernel_size=1),
            nn.Sigmoid(),
        )
        self.refine = nn.Sequential(
            nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        attention = self.gate(self.pool(x))
        refined = self.refine(x * attention)
        return x + refined


class VulnerabilityAwareClassifier(nn.Module):
    def __init__(self, in_channels, num_classes=19, reduction=8, dropout=0.1):
        super().__init__()
        self.attention = ResidualChannelAttention(in_channels, reduction=reduction)
        self.dropout = nn.Dropout2d(dropout)
        self.classifier = nn.Conv2d(in_channels, num_classes, kernel_size=1)

    def forward(self, x):
        x = self.attention(x)
        x = self.dropout(x)
        return self.classifier(x)

print("✓ Custom classifier definition restored.")



## 5. Load the original and modified models




In [ ]:

processor = AutoImageProcessor.from_pretrained(BASE_CHECKPOINT)

# Original fixed baseline
baseline_model = SegformerForSemanticSegmentation.from_pretrained(BASE_CHECKPOINT)
baseline_model.to(DEVICE).eval()

# Modified model
modified_model = SegformerForSemanticSegmentation.from_pretrained(BASE_CHECKPOINT)
old_classifier = modified_model.decode_head.classifier
in_channels = old_classifier.in_channels

modified_model.decode_head.classifier = VulnerabilityAwareClassifier(
    in_channels=in_channels,
    num_classes=NUM_CLASSES,
    reduction=mod_config['attention_head']['reduction'],
    dropout=mod_config['attention_head']['dropout'],
)

state = torch.load(STATE_PATH, map_location='cpu')
load_result = modified_model.load_state_dict(state, strict=True)
print("Modified state loaded:", load_result)

modified_model.to(DEVICE).eval()

print("Baseline parameters:", sum(p.numel() for p in baseline_model.parameters()))
print("Modified parameters:", sum(p.numel() for p in modified_model.parameters()))
print("✓ Both models loaded on:", DEVICE)


In [ ]:

# Quick architecture/checkpoint verification
assert isinstance(modified_model.decode_head.classifier, VulnerabilityAwareClassifier)
assert not isinstance(baseline_model.decode_head.classifier, VulnerabilityAwareClassifier)

baseline_model.eval()
modified_model.eval()

print("Baseline classifier:", type(baseline_model.decode_head.classifier).__name__)
print("Modified classifier:", type(modified_model.decode_head.classifier).__name__)
print("✓ Baseline and modified architectures are distinct as intended.")



## 6. Evaluation helpers



In [ ]:

def iou_from_confusion(cm):
    cm = np.asarray(cm, dtype=np.float64)
    tp = np.diag(cm)
    gt = cm.sum(axis=1)
    pred = cm.sum(axis=0)
    union = gt + pred - tp

    iou = np.full(NUM_CLASSES, np.nan, dtype=np.float64)
    valid = union > 0
    iou[valid] = tp[valid] / union[valid]
    return iou


def confusion_from_arrays(gt, pred, valid_mask):
    gt_v = gt[valid_mask].astype(np.int64)
    pred_v = pred[valid_mask].astype(np.int64)

    bins = np.bincount(
        NUM_CLASSES * gt_v + pred_v,
        minlength=NUM_CLASSES * NUM_CLASSES,
    )
    return bins.reshape(NUM_CLASSES, NUM_CLASSES).astype(np.int64)


@torch.inference_mode()
def predict_original_resolution(model, image, gt_shape):
    inputs = processor(images=image, return_tensors='pt')
    pixel_values = inputs['pixel_values'].to(DEVICE, non_blocking=True)

    with torch.amp.autocast('cuda', enabled=True):
        logits = model(pixel_values=pixel_values).logits

    logits = F.interpolate(
        logits,
        size=gt_shape,
        mode='bilinear',
        align_corners=False,
    )
    pred = logits.argmax(dim=1)[0].cpu().numpy().astype(np.int64)
    return pred



## 7. Run paired validation evaluation



In [ ]:

baseline_condition_cms = {
    c: np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=np.int64)
    for c in CONDITIONS
}
modified_condition_cms = {
    c: np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=np.int64)
    for c in CONDITIONS
}

audit_rows = []

for idx, row in tqdm(val_manifest.iterrows(), total=len(val_manifest), desc='Evaluating 406 ACDC val images'):
    image = Image.open(row.rgb_path).convert('RGB')
    gt = np.array(Image.open(row.gt_path), dtype=np.int64)
    inv = np.array(Image.open(row.inv_path), dtype=np.uint8)

    assert gt.shape == inv.shape, f"GT/inv shape mismatch: {row.scene_key}"

    semantic_valid = (gt >= 0) & (gt < NUM_CLASSES)
    valid = semantic_valid & (inv == 0)

    baseline_pred = predict_original_resolution(baseline_model, image, gt.shape)
    modified_pred = predict_original_resolution(modified_model, image, gt.shape)

    assert baseline_pred.shape == gt.shape
    assert modified_pred.shape == gt.shape

    baseline_cm = confusion_from_arrays(gt, baseline_pred, valid)
    modified_cm = confusion_from_arrays(gt, modified_pred, valid)

    baseline_condition_cms[row.condition] += baseline_cm
    modified_condition_cms[row.condition] += modified_cm

    # Save paired per-image matrices for later paired bootstrap analysis.
    np.savez_compressed(
        PER_IMAGE_CACHE_DIR / f"{row.condition}__{row.scene_key}.npz",
        condition=np.array(row.condition),
        scene_key=np.array(row.scene_key),
        baseline_confusion=baseline_cm,
        modified_confusion=modified_cm,
        valid_pixels=np.array(int(valid.sum()), dtype=np.int64),
        total_pixels=np.array(int(gt.size), dtype=np.int64),
    )

    audit_rows.append({
        'condition': row.condition,
        'scene_key': row.scene_key,
        'total_pixels': int(gt.size),
        'semantic_valid_pixels': int(semantic_valid.sum()),
        'inv_valid_pixels': int((inv == 0).sum()),
        'evaluation_valid_pixels': int(valid.sum()),
        'evaluation_valid_fraction': float(valid.mean()),
    })

    del baseline_pred, modified_pred, baseline_cm, modified_cm
    if (idx + 1) % 50 == 0:
        torch.cuda.empty_cache()

audit_df = pd.DataFrame(audit_rows)
audit_df.to_csv(EVAL_RESULTS_DIR / 'validation_mask_audit.csv', index=False)

print("✓ Evaluation complete.")
print("Per-image paired caches:", len(list(PER_IMAGE_CACHE_DIR.glob('*.npz'))))
assert len(list(PER_IMAGE_CACHE_DIR.glob('*.npz'))) == 406



## 8. Condition-wise mIoU: Original vs Modified


In [ ]:

summary_rows = []
per_class_rows = []

for condition in CONDITIONS:
    b_iou = iou_from_confusion(baseline_condition_cms[condition])
    m_iou = iou_from_confusion(modified_condition_cms[condition])

    b_miou = float(np.nanmean(b_iou))
    m_miou = float(np.nanmean(m_iou))

    summary_rows.append({
        'condition': condition,
        'images': int((val_manifest['condition'] == condition).sum()),
        'baseline_mIoU': b_miou,
        'modified_mIoU': m_miou,
        'mIoU_improvement': m_miou - b_miou,
    })

    for class_id, class_name in enumerate(CLASS_NAMES):
        per_class_rows.append({
            'condition': condition,
            'class_id': class_id,
            'class_name': class_name,
            'baseline_IoU': b_iou[class_id],
            'modified_IoU': m_iou[class_id],
            'IoU_improvement': m_iou[class_id] - b_iou[class_id]
                if np.isfinite(b_iou[class_id]) and np.isfinite(m_iou[class_id])
                else np.nan,
        })

summary_df = pd.DataFrame(summary_rows)
per_class_df = pd.DataFrame(per_class_rows)

summary_df.to_csv(EVAL_RESULTS_DIR / 'baseline_vs_modified_condition_miou.csv', index=False)
per_class_df.to_csv(EVAL_RESULTS_DIR / 'baseline_vs_modified_per_class_iou.csv', index=False)

display(summary_df.style.format({
    'baseline_mIoU': '{:.4f}',
    'modified_mIoU': '{:.4f}',
    'mIoU_improvement': '{:+.4f}',
}))


In [ ]:

# Sanity check against the previously established baseline ACDC validation scores.
# Small floating-point differences are acceptable; large differences indicate a protocol mismatch.
EXPECTED_BASELINE = {
    'fog': 0.6967,
    'rain': 0.5002,
    'snow': 0.4875,
    'night': 0.3294,
}

print("Baseline protocol sanity check:")
for row in summary_df.itertuples(index=False):
    expected = EXPECTED_BASELINE[row.condition]
    diff = row.baseline_mIoU - expected
    print(
        f"{row.condition:>5}: current={row.baseline_mIoU:.4f} | "
        f"previous={expected:.4f} | diff={diff:+.4f}"
    )
    assert abs(diff) < 0.003, (
        f"Baseline {row.condition} differs too much from Notebook 03. "
        "Stop and check preprocessing/masking before interpreting modified results."
    )

print("✓ Baseline matches the established evaluation protocol.")



## 9. Per-class before/after comparison


In [ ]:

# Pivot improvement for all 19 classes × 4 conditions
improvement_pivot = per_class_df.pivot(
    index='class_name',
    columns='condition',
    values='IoU_improvement'
).reindex(index=CLASS_NAMES, columns=CONDITIONS)

display(
    improvement_pivot.style
    .format('{:+.4f}')
    .background_gradient(cmap='RdYlGn', axis=None)
)

improvement_pivot.to_csv(EVAL_RESULTS_DIR / 'per_class_iou_improvement_matrix.csv')


In [ ]:

# Largest improvements and largest regressions
ranked = per_class_df.dropna(subset=['IoU_improvement']).copy()

print("Top 15 improvements:")
display(
    ranked.sort_values('IoU_improvement', ascending=False)
    .head(15)
    [['condition','class_name','baseline_IoU','modified_IoU','IoU_improvement']]
    .style.format({
        'baseline_IoU': '{:.4f}',
        'modified_IoU': '{:.4f}',
        'IoU_improvement': '{:+.4f}',
    })
)

print("\nTop 15 regressions:")
display(
    ranked.sort_values('IoU_improvement', ascending=True)
    .head(15)
    [['condition','class_name','baseline_IoU','modified_IoU','IoU_improvement']]
    .style.format({
        'baseline_IoU': '{:.4f}',
        'modified_IoU': '{:.4f}',
        'IoU_improvement': '{:+.4f}',
    })
)



## 10. Did the diagnosis-targeted classes improve?

Notebook 09 assigned larger loss weights to classes with:
- larger positive matched loss,
- more positive bootstrap direction,
- stronger both-side class support.

Here we merge those **train-only diagnosis weights** with the **untouched validation improvement**.


In [ ]:

target_comparison_df = per_class_df.merge(
    weight_df[
        ['condition','class_id','class_name','matched_change',
         'positive_resample_fraction','pairs_with_class_both',
         'raw_vulnerability_score','loss_weight']
    ],
    on=['condition','class_id','class_name'],
    how='left'
)

target_comparison_df['was_emphasized'] = target_comparison_df['loss_weight'] > 1.05

target_comparison_df.to_csv(
    EVAL_RESULTS_DIR / 'diagnosis_targets_vs_validation_improvement.csv',
    index=False
)

print("Most emphasized condition/class targets and what happened on validation:")
display(
    target_comparison_df
    .sort_values('loss_weight', ascending=False)
    .head(20)
    [['condition','class_name','loss_weight','matched_change',
      'positive_resample_fraction','pairs_with_class_both',
      'baseline_IoU','modified_IoU','IoU_improvement']]
    .style.format({
        'loss_weight': '{:.3f}',
        'matched_change': '{:+.4f}',
        'positive_resample_fraction': '{:.1%}',
        'baseline_IoU': '{:.4f}',
        'modified_IoU': '{:.4f}',
        'IoU_improvement': '{:+.4f}',
    })
)

emphasized = target_comparison_df[
    target_comparison_df['was_emphasized']
    & target_comparison_df['IoU_improvement'].notna()
]

print(f"Emphasized class-condition entries evaluated: {len(emphasized)}")
print("Mean IoU improvement among emphasized entries:",
      f"{emphasized['IoU_improvement'].mean():+.4f}")
print("Fraction of emphasized entries that improved:",
      f"{(emphasized['IoU_improvement'] > 0).mean():.1%}")



## 11. Class-group comparison

This uses the same project-defined groups from Notebook 07:
- Static Thin
- Dynamic Small
- Large/Structural

The group value here is the **unweighted mean of the finite class IoUs** within each group.


In [ ]:

group_rows = []

for condition in CONDITIONS:
    cond_df = per_class_df[per_class_df['condition'] == condition].set_index('class_name')

    for group_name, members in GROUPS.items():
        sub = cond_df.loc[members]
        b = float(np.nanmean(sub['baseline_IoU'].values))
        m = float(np.nanmean(sub['modified_IoU'].values))

        group_rows.append({
            'condition': condition,
            'group': group_name,
            'baseline_group_IoU': b,
            'modified_group_IoU': m,
            'group_IoU_improvement': m - b,
        })

group_df = pd.DataFrame(group_rows)
group_df.to_csv(EVAL_RESULTS_DIR / 'baseline_vs_modified_class_groups.csv', index=False)

display(
    group_df.style.format({
        'baseline_group_IoU': '{:.4f}',
        'modified_group_IoU': '{:.4f}',
        'group_IoU_improvement': '{:+.4f}',
    })
)



## 12. Visual summaries


In [ ]:

# Condition-wise mIoU comparison
plot_df = summary_df.set_index('condition')[['baseline_mIoU','modified_mIoU']]
ax = plot_df.plot(kind='bar', figsize=(8,5))
ax.set_ylabel('mIoU')
ax.set_xlabel('Condition')
ax.set_title('ACDC Validation: Original vs Modified SegFormer-B2')
ax.set_ylim(0, max(0.75, float(plot_df.max().max()) + 0.05))
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(EVAL_RESULTS_DIR / 'condition_miou_original_vs_modified.png',
            dpi=250, bbox_inches='tight')
plt.show()


In [ ]:

# Per-class improvement heatmap
fig, ax = plt.subplots(figsize=(8, 9))
data = improvement_pivot.values

im = ax.imshow(data, aspect='auto', cmap='RdYlGn')
ax.set_xticks(range(len(CONDITIONS)), CONDITIONS)
ax.set_yticks(range(len(CLASS_NAMES)), CLASS_NAMES)
ax.set_title('Modified − Original Per-Class IoU on ACDC Validation')

for i in range(data.shape[0]):
    for j in range(data.shape[1]):
        if np.isfinite(data[i, j]):
            ax.text(j, i, f'{data[i,j]:+.2f}',
                    ha='center', va='center', fontsize=8)

fig.colorbar(im, ax=ax, label='IoU improvement')
plt.tight_layout()
plt.savefig(EVAL_RESULTS_DIR / 'per_class_improvement_heatmap.png',
            dpi=250, bbox_inches='tight')
plt.show()


In [ ]:

# Class-group improvement
group_pivot = group_df.pivot(
    index='condition',
    columns='group',
    values='group_IoU_improvement'
).reindex(CONDITIONS)

ax = group_pivot.plot(kind='bar', figsize=(9,5))
ax.axhline(0, linewidth=1)
ax.set_ylabel('Modified − Original group IoU')
ax.set_xlabel('Condition')
ax.set_title('Class-Group IoU Improvement on ACDC Validation')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(EVAL_RESULTS_DIR / 'class_group_improvement.png',
            dpi=250, bbox_inches='tight')
plt.show()



## 13. Automatic interpretation summary

This cell gives a **descriptive** summary only. Statistical uncertainty of the *before/after improvement* should be assessed with the paired per-image caches in the next analysis notebook.


In [ ]:

best_condition = summary_df.loc[summary_df['mIoU_improvement'].idxmax()]
worst_condition = summary_df.loc[summary_df['mIoU_improvement'].idxmin()]

best_class = ranked.loc[ranked['IoU_improvement'].idxmax()]
worst_class = ranked.loc[ranked['IoU_improvement'].idxmin()]

print("=== NOTEBOOK 10 SUMMARY ===")
print(
    f"Largest condition-level mIoU improvement: "
    f"{best_condition['condition']} {best_condition['mIoU_improvement']:+.4f}"
)
print(
    f"Smallest / worst condition-level change: "
    f"{worst_condition['condition']} {worst_condition['mIoU_improvement']:+.4f}"
)
print(
    f"Largest class-condition improvement: "
    f"{best_class['condition']} / {best_class['class_name']} "
    f"{best_class['IoU_improvement']:+.4f}"
)
print(
    f"Largest class-condition regression: "
    f"{worst_class['condition']} / {worst_class['class_name']} "
    f"{worst_class['IoU_improvement']:+.4f}"
)

if len(emphasized):
    print(
        f"Among diagnosis-emphasized entries, "
        f"{(emphasized['IoU_improvement'] > 0).mean():.1%} improved; "
        f"mean improvement = {emphasized['IoU_improvement'].mean():+.4f}."
    )

print("\nImportant: these are validation point estimates.")
print("Use the saved 406 paired per-image confusion files for paired bootstrap uncertainty of improvement.")


In [ ]:
# Figure (a): Condition-wise mIoU comparison with values above bars

plot_df = summary_df.set_index('condition')[
    ['baseline_mIoU', 'modified_mIoU']
]

fig, ax = plt.subplots(figsize=(8, 5))

plot_df.plot(
    kind='bar',
    ax=ax
)

ax.set_ylabel('mIoU')
ax.set_xlabel('Condition')
ax.set_title('ACDC Validation: Original vs Modified SegFormer-B2')

ax.set_ylim(
    0,
    max(0.78, float(plot_df.max().max()) + 0.08)
)

plt.xticks(rotation=0)

# Add exact values above each bar
for container in ax.containers:
    ax.bar_label(
        container,
        fmt='%.3f',
        padding=3,
        fontsize=9
    )

# Panel label
ax.text(
    -0.08, 1.05,
    '(a)',
    transform=ax.transAxes,
    fontsize=13,
    fontweight='bold',
    va='top',
    ha='left'
)

plt.tight_layout()

figure_path = (
    EVAL_RESULTS_DIR
    / 'condition_miou_original_vs_modified_a.png'
)

plt.savefig(
    figure_path,
    dpi=300,
    bbox_inches='tight'
)

plt.show()

print("Saved:", figure_path)


# Figure (b): Paired bootstrap with numerical values

plot_df = miou_boot_df.copy()

x = np.arange(len(plot_df))

bootstrap_means = plot_df['bootstrap_mean'].values
ci_low = plot_df['ci_low'].values
ci_high = plot_df['ci_high'].values

point_estimates = plot_df[
    'original_sample_improvement'
].values

yerr = np.vstack([
    bootstrap_means - ci_low,
    ci_high - bootstrap_means
])

fig, ax = plt.subplots(figsize=(8, 5))

# Bootstrap mean + 95% CI
ax.errorbar(
    x,
    bootstrap_means,
    yerr=yerr,
    fmt='o',
    capsize=5,
    label='Bootstrap mean and 95% interval'
)

# Original-sample estimates
ax.scatter(
    x,
    point_estimates,
    marker='x',
    s=70,
    label='Original-sample improvement'
)

# Add numerical ORIGINAL-SAMPLE improvement beside each point
for i, value in enumerate(point_estimates):
    ax.annotate(
        f'{value:+.3f}',
        (x[i], value),
        xytext=(8, 5),
        textcoords='offset points',
        fontsize=9,
        fontweight='bold'
    )

ax.axhline(
    0,
    linewidth=1
)

ax.set_xticks(x)
ax.set_xticklabels(plot_df['condition'])

ax.set_ylabel('Modified − Original mIoU')
ax.set_xlabel('Condition')
ax.set_title('Paired Bootstrap of Model Improvement')

ax.legend()

# Panel label
ax.text(
    -0.08, 1.05,
    '(b)',
    transform=ax.transAxes,
    fontsize=13,
    fontweight='bold',
    va='top',
    ha='left'
)

plt.tight_layout()

figure_path = (
    BOOT_DIR
    / 'miou_improvement_bootstrap_intervals_b.png'
)

plt.savefig(
    figure_path,
    dpi=300,
    bbox_inches='tight'
)

plt.show()

print("Saved:", figure_path)